In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [11]:
X = torch.rand(1, 28, 28, device=device)
# print(X)
logits = model(X)
# print(logits)
pred_probab = nn.Softmax(dim=1)(logits)
# print(pred_probab)
y_pred = pred_probab.argmax(1)
print(y_pred.item())
print(f"Predicted class: {y_pred}")

7
Predicted class: tensor([7])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [12]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [13]:
# 使用flatten将维度压缩
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [14]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [16]:
# 激活层前后形状不变
print(f"Before ReLU: {hidden1}\n\n")
print(hidden1.shape)
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")
print(hidden1.shape)

Before ReLU: tensor([[0.3100, 0.0000, 0.2432, 0.3717, 0.0000, 0.3370, 0.0000, 0.0000, 0.3262,
         0.0000, 0.0000, 0.1060, 0.3229, 0.0000, 0.0269, 0.1678, 0.2453, 0.4615,
         0.0439, 0.0000],
        [0.1414, 0.0000, 0.6518, 0.2782, 0.0000, 0.0000, 0.0960, 0.0000, 0.0088,
         0.0000, 0.0000, 0.0000, 0.3112, 0.0000, 0.2696, 0.0000, 0.1834, 0.0905,
         0.0000, 0.1829],
        [0.3743, 0.0000, 0.4438, 0.3654, 0.0008, 0.2583, 0.3289, 0.0000, 0.3244,
         0.0000, 0.3340, 0.0242, 0.6539, 0.0000, 0.4913, 0.1207, 0.4360, 0.3799,
         0.0000, 0.0000]], grad_fn=<ReluBackward0>)


torch.Size([3, 20])
After ReLU: tensor([[0.3100, 0.0000, 0.2432, 0.3717, 0.0000, 0.3370, 0.0000, 0.0000, 0.3262,
         0.0000, 0.0000, 0.1060, 0.3229, 0.0000, 0.0269, 0.1678, 0.2453, 0.4615,
         0.0439, 0.0000],
        [0.1414, 0.0000, 0.6518, 0.2782, 0.0000, 0.0000, 0.0960, 0.0000, 0.0088,
         0.0000, 0.0000, 0.0000, 0.3112, 0.0000, 0.2696, 0.0000, 0.1834, 0.0905,
         0.00

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [20]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.0028, -0.0902, -0.0081, -0.0740,  0.1596, -0.1853,  0.0596,  0.1291,
          0.1589,  0.2233],
        [-0.0516, -0.2281, -0.0684, -0.1428,  0.1820, -0.1241, -0.0212, -0.0084,
          0.1821,  0.1353],
        [ 0.0167, -0.1412, -0.0744, -0.1203,  0.0770, -0.0717, -0.0223,  0.2003,
          0.0615,  0.1907]], grad_fn=<AddmmBackward0>)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [21]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.0958, 0.0873, 0.0948, 0.0888, 0.1121, 0.0794, 0.1014, 0.1087, 0.1120,
         0.1195],
        [0.0955, 0.0800, 0.0939, 0.0872, 0.1206, 0.0888, 0.0985, 0.0997, 0.1206,
         0.1151],
        [0.0998, 0.0853, 0.0911, 0.0871, 0.1060, 0.0914, 0.0960, 0.1200, 0.1044,
         0.1188]], grad_fn=<SoftmaxBackward0>)


Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [22]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0079,  0.0116, -0.0216,  ...,  0.0135,  0.0112,  0.0314],
        [-0.0297, -0.0080, -0.0135,  ...,  0.0249,  0.0209,  0.0171]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0152, 0.0106], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0176, -0.0222,  0.0033,  ..., -0.0119,  0.0076,  0.0350],
        [-0.0004,  0.0139,  0.0190,  ..., -0.0047,  0.0422,  0.0253]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
